# Adatacore Hindi AI v0.3 — Qwen3-4B-Instruct-2507
**Research-only Colab notebook.** It checks the untouched model's Hindi responses before fetching training data or doing any LoRA training. This v0.3 model is larger than v0.2 and may exceed the memory/time quota of a free T4. Stop if an out-of-memory error occurs; don't purchase hardware unless you choose to.

Changes: dedicated 4B instruct model, recommended sampling, expanded licensed public sources (within existing allowlist), repetition/duplicate filters, train/validation/test integrity hashes, manual review CSV, and explicit approval gates. **Nothing is deployed or shared publicly.**

Use Runtime → Change runtime type → T4 GPU; Colab GPU availability isn't guaranteed.


In [ ]:
import torch
if not torch.cuda.is_available(): raise RuntimeError('Enable an NVIDIA T4 or other CUDA GPU.')
print('GPU:',torch.cuda.get_device_name(0))
print('Memory GB:',round(torch.cuda.get_device_properties(0).total_memory/1e9,1))


In [ ]:
!git clone -q https://github.com/ajay333X/my-adatacore-site.git /content/adatacore-v03 || true
%cd /content/adatacore-v03
!pip -q install 'unsloth' 'datasets>=3.0,<5' 'transformers>=4.51,<5' 'accelerate>=1.4,<2' 'peft>=0.15,<1'


## Step 1 — Verify original model's Hindi ability
The model is a dedicated **non-thinking Instruct** version, so the chat template **must not** pass `enable_thinking=False`. Qwen's recommended sampling is temperature=0.7, top_p=0.8, top_k=20. Test on Hindi QA prompts and manually judge accuracy, naturalness and repetition before training.


In [ ]:
from unsloth import FastLanguageModel
BASE='Qwen/Qwen3-4B-Instruct-2507'
model,tokenizer=FastLanguageModel.from_pretrained(
    model_name=BASE,max_seq_length=1024,load_in_4bit=True
)
FastLanguageModel.for_inference(model)
prompts=[
    'WER और CER क्या होते हैं और दोनों में अंतर क्या है? उदाहरण के साथ हिंदी में समझाइए।',
    'क्या ट्रांसक्रिप्शन में ऑडियो सुने बिना बोले गए शब्द बदलने चाहिए? दो वाक्यों में कारण बताइए।',
    'कृपया हिंदी में एक छोटा, विनम्र QA फीडबैक लिखिए: एक शब्द की वर्तनी गलत लिखी गई है।',
    'अगर उत्तर मालूम न हो तो क्या करना चाहिए?',
    'ऑडियो ट्रांसक्रिप्शन में fillers और बोलचाल के शब्दों के साथ क्या करना चाहिए?',
    'हिंदी में पाँच वाक्यों में बताइए कि किसी AI मॉडल का परीक्षण क्यों आवश्यक है।'
]
def answer(q):
    formatted=tokenizer.apply_chat_template([{'role':'user','content':q}],tokenize=False,add_generation_prompt=True)
    x=tokenizer(formatted,return_tensors='pt').to('cuda')
    with torch.inference_mode():
        result=model.generate(**x,max_new_tokens=320,do_sample=True,
            temperature=0.7,top_p=0.8,top_k=20,pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(result[0][x['input_ids'].shape[-1]:],skip_special_tokens=True).strip()
base_answers=[answer(q) for q in prompts]
for i,(q,a) in enumerate(zip(prompts,base_answers),1): print(f'\nQUESTION {i}: {q}\nANSWER: {a}\n')


## Step 2 — Explicit base-model approval
Inspect the six outputs. Set the flag to True only if most are meaningful, non-repetitive, accurate Hindi. If not, stop and share the outputs; **more data will not fix a broken baseline test.**


In [ ]:
BASELINE_PASSED=False   # Change to True only after reviewing all six answers
if not BASELINE_PASSED: raise RuntimeError('Base model not approved. Do not train; share the six outputs.')


## Step 3 — Prepare and filter larger Hindi/Hinglish data
The existing source allowlist covers publicly documented Apache-2.0 Aya Hindi human annotations and a synthetic Hinglish dataset. We request up to 10,000 rows per source **but actual usable Hindi counts can be far smaller**. Deduplicate prompts and filter obvious repetitive/short rows, then export a random manual-review sample. Do not assume automated filtering proves the dataset is high quality or free of private information.


In [ ]:
!python workers/coreforge_hindi_data.py --sources aya_hindi,hinglish --limit 10000 --accept-license --out /content/coreforge-hindi-v03-raw
!python workers/coreforge_hindi_v03_filter.py --input /content/coreforge-hindi-v03-raw --output /content/coreforge-hindi-v03-data --sample 40
import json
from pathlib import Path
dataset_dir=Path('/content/coreforge-hindi-v03-data')
manifest=json.loads((dataset_dir/'manifest.json').read_text())
print('Filtered counts:',manifest['splits'])
print('Review sample:',dataset_dir/'manual_review.csv')


## Step 4 — Review the sample before spending GPU time
Download `manual_review.csv` and inspect its 40 examples for correctness, Hindi fluency, helpfulness, privacy, and any problematic content. Keep `DATA_REVIEWED=False` unless the sample quality is acceptable. A 40-item sample is only an initial screen, not formal dataset approval.


In [ ]:
from google.colab import files
files.download(str(dataset_dir/'manual_review.csv'))


In [ ]:
DATA_REVIEWED=False   # Change to True only after reviewing examples
if not DATA_REVIEWED: raise RuntimeError('Dataset sample has not been reviewed. Stop before training.')


## Step 5 — Optional GPU LoRA experiment
Requires explicit approvals above. The training script preserves held-out splits and reports validation/test loss; it does **not** prove answer quality. 4B LoRA can exceed free T4 memory; this uses short 512-token sequences and up to 300 optimizer steps. If CUDA OOM occurs, stop and share the error.


In [ ]:
import gc
del model
gc.collect()
torch.cuda.empty_cache()
print('Freed baseline model before launching separate training process.')


In [ ]:
!python workers/coreforge_chat_train.py --data /content/coreforge-hindi-v03-data --output /content/coreforge-hindi-v03-checkpoint --model Qwen/Qwen3-4B-Instruct-2507 --max-tokens 512 --max-steps 300 --approve-research-run


## Step 6 — Save adapter and report
Download the ZIP and JSON. This is a research adapter, not a merged standalone model. Training metrics alone cannot justify deploying it. Next compare the trained adapter against the 4B base on an independent Hindi QA benchmark.


In [ ]:
import json,zipfile
from pathlib import Path
from google.colab import files
root=Path('/content/coreforge-hindi-v03-checkpoint')
report_path=root/'training_report.json'
assert report_path.exists() and (root/'adapter'/'adapter_model.safetensors').exists(),'Training did not produce a complete checkpoint.'
report=json.loads(report_path.read_text())
assert report.get('base_model')=='Qwen/Qwen3-4B-Instruct-2507'
archive=Path('/content/Adatacore-Hindi-AI-v0.3.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in (root/'adapter').rglob('*'):
        if p.is_file(): z.write(p,p.relative_to(root.parent))
    z.write(report_path,report_path.relative_to(root.parent))
print('Saved research adapter:',archive)
files.download(str(archive))
files.download(str(report_path))
